In [ ]:
#| default_exp storyboard

# storyboard

> Storyboard generation: convert story chunks into comic scenes and panels.
>
> Each chunk of the story becomes one `Scene`. The LLM decides panel count,
> camera angles, dialogue and actions. Visual prompts are assembled by combining
> the style prefix, location prompt, character prompts and action description.
> Results are saved to `storyboard.json` and resumed automatically.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import json
from pathlib import Path

from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TextColumn

from manhualizer.config import PipelineConfig
from manhualizer.llm import LLMClient, chunk_story
from manhualizer.models import (
    DialogueBubble, Panel, Scene, Storyboard, StoryAnalysis,
)
from manhualizer.prompts import TemplateSet

_console = Console()

## Prompt Assembly Helpers

In [ ]:
#| export
def _character_lookup(analysis: StoryAnalysis) -> dict[str, str]:
    """Build a name → reference_image_prompt map (case-insensitive keys)."""
    return {
        c.name.lower(): c.reference_image_prompt
        for c in analysis.characters
        if c.reference_image_prompt
    }


def _location_lookup(analysis: StoryAnalysis) -> dict[str, str]:
    """Build a name → visual_prompt map (case-insensitive keys)."""
    return {
        l.name.lower(): l.visual_prompt
        for l in analysis.locations
        if l.visual_prompt
    }


def _dialogue_instructions(panel_data: dict) -> str:
    """Build image prompt instructions for dialogue bubbles.

    Tells the image model to render text inside speech/thought/caption bubbles
    directly in the image so panels are self-contained for vertical scrolling.

    Manhwa bubble visual guide:
    - speech  → oval bubble, single pointed tail toward ONE speaker only
    - shout   → jagged spiky starburst bubble, single tail toward ONE speaker only
    - whisper → small oval bubble with dashed border, single tail toward ONE speaker only
    - thought → cloud shape with dotted trail toward speaker
    - caption → rectangular box with black border at top or bottom panel edge
    - sfx     → large bold stylized sound effect text
    """
    dialogue = panel_data.get("dialogue", [])
    if not dialogue:
        return ""

    parts = []
    for d in dialogue:
        bubble = d.get("bubble_type", "speech")
        speaker = d.get("speaker", "")
        text = d.get("text", "")
        if not text:
            continue

        if bubble == "caption" or speaker == "narration":
            parts.append(
                f'rectangular narration box with black border and white background'
                f', placed at the top or bottom edge of the panel, text: "{text}"'
            )
        elif bubble == "shout":
            parts.append(
                f'jagged spiky starburst-shaped shout bubble with thick black border and bold text'
                f', single tail pointing only toward {speaker}, text: "{text}"'
            )
        elif bubble == "whisper":
            parts.append(
                f'small oval speech bubble with dashed border and small italic text'
                f', single tail pointing only toward {speaker}, text: "{text}"'
            )
        elif bubble == "thought":
            parts.append(
                f'cloud-shaped thought bubble with dotted border trailing toward {speaker}'
                f', text: "{text}"'
            )
        elif bubble == "sfx":
            parts.append(f'large bold stylized sound effect text "{text}"')
        else:
            parts.append(
                f'oval speech bubble with a single pointed tail directed only toward {speaker}'
                f', white fill, black border, text: "{text}"'
            )

    if not parts:
        return ""
    return "Contains: " + "; ".join(parts) + "."


def _assemble_visual_prompt(
    panel_data: dict,
    templates: TemplateSet,
    char_prompts: dict[str, str],
    loc_prompts: dict[str, str],
) -> str:
    """Assemble the final image-gen prompt for a panel.

    If the LLM already produced a complete visual_prompt, use it directly.
    Otherwise build from parts using the style template.
    Dialogue bubble instructions are always appended so panels are self-contained.
    """
    dialogue_part = _dialogue_instructions(panel_data)

    # Use LLM-provided prompt if present and substantial
    llm_prompt = panel_data.get("visual_prompt", "").strip()
    if llm_prompt and len(llm_prompt) > 20:
        # Ensure style prefix is prepended if missing
        style = templates.style_prefix
        if style and not llm_prompt.lower().startswith(style[:15].lower()):
            base = f"{style}, {llm_prompt}"
        else:
            base = llm_prompt
        return f"{base}. {dialogue_part}" if dialogue_part else base

    # Build from parts
    location_name = panel_data.get("location", "").lower()
    location_prompt = loc_prompts.get(location_name, location_name)

    chars_present = [c.lower() for c in panel_data.get("characters_present", [])]
    char_prompt_parts = [char_prompts[c] for c in chars_present if c in char_prompts]
    character_prompts = ", ".join(char_prompt_parts) if char_prompt_parts else "no characters"

    action = panel_data.get("action_description", "")
    mood = panel_data.get("mood", "")
    camera = panel_data.get("camera_angle", "")

    style_prefix = templates.style_prefix
    parts = [p for p in [style_prefix, location_prompt, character_prompts, action, mood, camera] if p]
    base = ", ".join(parts)
    return f"{base}. {dialogue_part}" if dialogue_part else base

## Parsing LLM Output

In [ ]:
#| export
def _parse_panel(
    data: dict,
    panel_number: int,
    scene_id: str,
    templates: TemplateSet,
    char_prompts: dict[str, str],
    loc_prompts: dict[str, str],
) -> Panel:
    dialogue = [
        DialogueBubble(
            speaker=d.get("speaker", ""),
            text=d.get("text", ""),
            bubble_type=d.get("bubble_type", "speech"),
        )
        for d in data.get("dialogue", [])
    ]
    visual_prompt = _assemble_visual_prompt(data, templates, char_prompts, loc_prompts)
    return Panel(
        panel_number=panel_number,
        scene_id=scene_id,
        characters_present=data.get("characters_present", []),
        location=data.get("location", ""),
        action_description=data.get("action_description", ""),
        visual_prompt=visual_prompt,
        dialogue=dialogue,
        mood=data.get("mood", ""),
        camera_angle=data.get("camera_angle", ""),
    )


def _parse_scene(
    data: dict,
    scene_index: int,
    panel_offset: int,
    chunk: str,
    templates: TemplateSet,
    char_prompts: dict[str, str],
    loc_prompts: dict[str, str],
) -> Scene:
    scene_id = data.get("scene_id", f"s{scene_index + 1}")
    panels = [
        _parse_panel(
            p, panel_offset + i + 1, scene_id,
            templates, char_prompts, loc_prompts,
        )
        for i, p in enumerate(data.get("panels", []))
    ]
    return Scene(
        scene_id=scene_id,
        title=data.get("title", f"Scene {scene_index + 1}"),
        source_chunk=chunk,
        panels=panels,
    )

## Main Storyboard Function

In [ ]:
#| export
def build_storyboard(
    story_text: str,
    analysis: StoryAnalysis,
    llm: LLMClient,
    templates: TemplateSet,
    config: PipelineConfig,
    output_path: Path,
) -> Storyboard:
    """Convert a story into a comic storyboard.

    Each chunk of the story text becomes one Scene. The LLM is given the full
    story analysis (characters, locations, style) so it can produce consistent,
    analysis-grounded panel descriptions and visual prompts.

    Visual prompts are assembled by prepending the style prefix from the active
    template and injecting the character/location reference prompts from the
    analysis. This keeps image generation prompts consistent across the whole comic.

    If `output_path` already exists and `config.resume` is True, the saved
    storyboard is returned without any LLM calls.

    Args:
        story_text: Full raw story text (same as passed to analyze_story).
        analysis: Output of analyze_story().
        llm: Configured LLMClient.
        templates: Active TemplateSet.
        config: PipelineConfig.
        output_path: Where to save storyboard.json.

    Returns:
        Storyboard with all scenes and panels populated.
    """
    output_path = Path(output_path)

    if config.resume and output_path.exists():
        _console.print(f"[dim]storyboard: resuming from {output_path}[/dim]")
        return Storyboard.model_validate_json(output_path.read_text())

    # Re-chunk the story with a tighter budget so each scene covers ~800 words.
    # This keeps the JSON response small enough to fit within LLM max_tokens.
    # (analyze may use larger chunks; storyboard needs smaller ones.)
    chunks = chunk_story(story_text, max_tokens=config.storyboard_chunk_tokens)
    char_prompts = _character_lookup(analysis)
    loc_prompts = _location_lookup(analysis)
    analysis_json = analysis.model_dump_json(indent=2, exclude_none=True)

    _console.print(f"[bold]storyboard:[/bold] generating {len(chunks)} scene(s)")

    scenes: list[Scene] = []
    panel_offset = 0

    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        console=_console,
        transient=True,
    ) as progress:
        task = progress.add_task("Building storyboard...", total=len(chunks))

        for i, chunk in enumerate(chunks):
            progress.update(task, description=f"Scene {i + 1}/{len(chunks)}...")
            scene_data = llm.complete_from_template(
                "storyboard.yml",
                "storyboard_prompt",
                as_json=True,
                max_tokens=16384,
                story_chunk=chunk,
                analysis=analysis_json,
                style_prefix=templates.style_prefix,
                scene_number=i + 1,
            )
            scene = _parse_scene(
                scene_data, i, panel_offset, chunk,
                templates, char_prompts, loc_prompts,
            )
            scenes.append(scene)
            panel_offset += len(scene.panels)
            progress.advance(task)

    storyboard = Storyboard(
        title=analysis.title,
        scenes=scenes,
        total_panels=panel_offset,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(storyboard.model_dump_json(indent=2, exclude_none=True))
    _console.print(f"[green]storyboard: saved to {output_path}[/green]")
    _console.print(
        f"  {len(scenes)} scene(s), {panel_offset} panel(s) total"
    )

    return storyboard

## Tests (no API calls)

In [ ]:
from manhualizer.storyboard import (
    _character_lookup, _location_lookup, _dialogue_instructions,
    _assemble_visual_prompt, _parse_scene,
)
from manhualizer.models import Character, Location, StoryAnalysis, Storyboard
from manhualizer.prompts import load_templates

templates = load_templates("default")

analysis = StoryAnalysis(
    title="The Dragon's Gift",
    synopsis="A young farmer discovers a dragon.",
    characters=[
        Character(
            name="Wei Chen",
            physical_description="Tall young man",
            personality="Determined",
            reference_image_prompt="young Chinese man, short black hair, determined expression",
        )
    ],
    locations=[
        Location(
            name="Village",
            description="Small farming village",
            visual_prompt="chinese village, rice paddies, wooden houses",
        )
    ],
    source_chunks=["Once upon a time, Wei Chen found the egg."],
)

char_prompts = _character_lookup(analysis)
loc_prompts = _location_lookup(analysis)

assert char_prompts["wei chen"] == "young Chinese man, short black hair, determined expression"
assert loc_prompts["village"] == "chinese village, rice paddies, wooden houses"

# Dialogue instructions
assert _dialogue_instructions({}) == ""
assert _dialogue_instructions({"dialogue": []}) == ""
di = _dialogue_instructions({"dialogue": [
    {"speaker": "Wei Chen", "text": "Hello!", "bubble_type": "speech"},
    {"speaker": "narration", "text": "He spoke.", "bubble_type": "caption"},
]})
assert "oval speech bubble" in di and "Wei Chen" in di and "Hello!" in di
assert "rectangular narration box" in di and "He spoke." in di

# Visual prompt assembly from parts (with dialogue)
panel_data = {
    "visual_prompt": "",  # force assembly from parts
    "location": "Village",
    "characters_present": ["Wei Chen"],
    "action_description": "discovers a glowing egg",
    "mood": "mysterious",
    "camera_angle": "wide shot",
    "dialogue": [{"speaker": "Wei Chen", "text": "What is this?", "bubble_type": "speech"}],
}
prompt = _assemble_visual_prompt(panel_data, templates, char_prompts, loc_prompts)
assert "manhua" in prompt
assert "rice paddies" in prompt
assert "young Chinese man" in prompt
assert "discovers a glowing egg" in prompt
assert "oval speech bubble" in prompt and "What is this?" in prompt

# Visual prompt passthrough when LLM provides a full prompt
panel_data2 = {
    "visual_prompt": "manhua style, forest clearing, Wei Chen standing tall, dramatic lighting",
    "location": "Village",
    "characters_present": ["Wei Chen"],
    "action_description": "stands",
    "mood": "dramatic",
    "dialogue": [],
}
prompt2 = _assemble_visual_prompt(panel_data2, templates, char_prompts, loc_prompts)
assert "forest clearing" in prompt2  # passed through

print("Visual prompt assembly OK")

In [ ]:
# Scene parsing test
scene_data = {
    "scene_id": "s1",
    "title": "Discovery",
    "panels": [
        {
            "visual_prompt": "",
            "characters_present": ["Wei Chen"],
            "location": "Village",
            "action_description": "walks through village",
            "mood": "peaceful",
            "camera_angle": "wide shot",
            "dialogue": [{"speaker": "Wei Chen", "text": "Another ordinary day.", "bubble_type": "speech"}],
        },
        {
            "visual_prompt": "manhua style, dark cave, Wei Chen kneeling, glowing egg in hands",
            "characters_present": ["Wei Chen"],
            "location": "Cave",
            "action_description": "finds the egg",
            "mood": "mysterious",
            "camera_angle": "close-up",
            "dialogue": [],
        },
    ]
}

scene = _parse_scene(scene_data, 0, 0, "chunk text", templates, char_prompts, loc_prompts)

assert scene.scene_id == "s1"
assert scene.title == "Discovery"
assert len(scene.panels) == 2
assert scene.panels[0].panel_number == 1
assert scene.panels[1].panel_number == 2
assert scene.panels[0].dialogue[0].speaker == "Wei Chen"
assert "rice paddies" in scene.panels[0].visual_prompt  # location prompt injected
assert "dark cave" in scene.panels[1].visual_prompt     # LLM prompt passed through

# Serialise round-trip
sb = Storyboard(title="Test", scenes=[scene], total_panels=2)
restored = Storyboard.model_validate_json(sb.model_dump_json())
assert len(restored.all_panels) == 2

print("Scene parsing OK")

In [ ]:
# Resume test
import tempfile
from pathlib import Path
from manhualizer.storyboard import build_storyboard
from manhualizer.config import PipelineConfig

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "storyboard.json"
    out.write_text(sb.model_dump_json())
    cfg = PipelineConfig(resume=True)
    result = build_storyboard("any text", analysis, llm=None, templates=templates, config=cfg, output_path=out)
    assert result.title == "Test"
    assert len(result.all_panels) == 2

print("Resume OK")
print("All storyboard tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()